# KrimbaGram · RVC voice conversion (Step 0) — RVC in a uv-managed Python 3.10 env

Kaggle's kernel is Python 3.12, but RVC (fairseq/hydra/omegaconf) only runs on **Python 3.10**. Kaggle has no conda, so we use **`uv`** to fetch a standalone Python 3.10 and run the conversion there as a subprocess. The 3.12 kernel just downloads files and plays the result.

**Before running:** GPU optional (uses CPU torch) · **Internet: On** · add **`HF_TOKEN`** secret (read token; repo is private). Run top to bottom. Cells 3–4 take a few minutes (one time).

In [ ]:
# 1) config
HF_REPO    = "cyttic/lecturer-ru-rvc"
PTH_FILE   = "lecturer_ru.pth"
INDEX_FILE = "lecturer_ru.index"
NAME       = "lecturer_ru"
RVC        = "/kaggle/working/rvc"
import sys; print("kernel python", sys.version.split()[0])

In [ ]:
# 2) clone RVC-WebUI
%cd /kaggle/working
![ -d rvc ] || git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI rvc

In [ ]:
# 3) build a Python 3.10 env with uv (downloads a standalone CPython 3.10; no conda/apt needed)
!pip install -q uv
!uv venv --seed --python 3.10 /kaggle/working/rvc310
!/kaggle/working/rvc310/bin/python --version

In [ ]:
# 4) install RVC deps INTO the 3.10 env  (env's own pip<24.1 handles fairseq's old omegaconf metadata)
PY = "/kaggle/working/rvc310/bin/python"
!{PY} -m pip install -q "pip<24.1"
!{PY} -m pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cpu
!{PY} -m pip install -q numpy==1.26.4 faiss-cpu praat-parselmouth pyworld torchcrepe ffmpeg-python av tensorboardX fairseq librosa soundfile scipy
!{PY} -c "import fairseq, faiss, torch; print('3.10 env OK | torch', torch.__version__)"

In [ ]:
# 5) base encoders (HuBERT + RMVPE) into the rvc/ tree
import os, urllib.request
os.makedirs(f"{RVC}/assets/hubert", exist_ok=True); os.makedirs(f"{RVC}/assets/rmvpe", exist_ok=True)
def grab(u, d):
    if not os.path.exists(d):
        print("downloading", d); urllib.request.urlretrieve(u, d)
grab("https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt", f"{RVC}/assets/hubert/hubert_base.pt")
grab("https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt", f"{RVC}/assets/rmvpe/rmvpe.pt")

In [ ]:
# 6) pull YOUR model from the private HF repo
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
import os, shutil
tok = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(f"{RVC}/assets/weights", exist_ok=True); os.makedirs(f"{RVC}/logs/{NAME}", exist_ok=True)
shutil.copy(hf_hub_download(HF_REPO, PTH_FILE,   token=tok), f"{RVC}/assets/weights/{NAME}.pth")
shutil.copy(hf_hub_download(HF_REPO, INDEX_FILE, token=tok), f"{RVC}/logs/{NAME}/added_{NAME}.index")
print("model + index in place")

In [ ]:
# 7) write the conversion script (runs inside the 3.10 env)
script = r"""
import os, sys, glob, time
import soundfile as sf
import torch as _t
_ol = _t.load
_t.load = lambda *a, **k: _ol(*a, **{**k, 'weights_only': False})
NAME = "lecturer_ru"
SRC  = "/kaggle/working/src.wav"
os.chdir("/kaggle/working/rvc")
sys.path.insert(0, "/kaggle/working/rvc")
os.environ["weight_root"] = "assets/weights"
os.environ["index_root"]  = "logs"
os.environ["rmvpe_root"]  = "assets/rmvpe"
if not os.path.exists(SRC):
    import librosa
    y, sr = librosa.load(librosa.example("libri1"), sr=16000, duration=12)
    sf.write(SRC, y, sr); print("made sample src:", round(len(y)/sr, 1), "s")
from infer.modules.vc.modules import VC
from configs.config import Config
_a = sys.argv; sys.argv = ["infer"]; config = Config(); sys.argv = _a
vc = VC(config); vc.get_vc(NAME + ".pth")
idx = (glob.glob("logs/" + NAME + "/added_*.index") or [""])[0]
print("index:", idx or "(none)")
def convert(ir, out):
    t = time.time()
    o = vc.vc_single(0, SRC, 0, None, "rmvpe", idx, "", ir, 3, 0, 0.25, 0.33)
    dt = time.time() - t
    sr, audio = o[1]; dur = len(audio) / sr
    print("index_rate=%s | out_sr=%d Hz | clip=%.1fs | convert=%.2fs | RTF=%.2fx" % (ir, sr, dur, dt, dt/dur))
    sf.write(out, audio, sr)
convert(0.66, "/kaggle/working/with_index.wav")
convert(0.0,  "/kaggle/working/no_index.wav")
print("DONE")
"""
open("/kaggle/working/convert.py", "w").write(script)
print("wrote /kaggle/working/convert.py")

In [ ]:
# 8) run the conversion in the 3.10 env  (makes a sample src.wav on first run)
!/kaggle/working/rvc310/bin/python /kaggle/working/convert.py

In [ ]:
# 9) listen
from IPython.display import Audio, display
import os
for label, p in [("INPUT","/kaggle/working/src.wav"),
                 ("WITH index","/kaggle/working/with_index.wav"),
                 ("NO index","/kaggle/working/no_index.wav")]:
    print(f"{label}: {p} (exists={os.path.exists(p)})")
    if os.path.exists(p): display(Audio(filename=p))